In [2]:
pip install torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [3]:
!mc ls s3/dimitri/stat_app/
!mc ls s3/dimitri/stat_app/CMMD2022

]11;?\mc: Configuration written to `/home/onyxia/.mc/config.json`. Please update your access credentials.
mc: Successfully created `/home/onyxia/.mc/share`.
mc: Initialized share uploads `/home/onyxia/.mc/share/uploads.json` file.
mc: Initialized share downloads `/home/onyxia/.mc/share/downloads.json` file.
[2026-01-27 14:08:49 UTC]    39B STANDARD .keep
[2026-01-29 12:26:03 UTC]     0B CMMD2022/
[2026-01-29 12:26:03 UTC]     0B miniddsm2/
]11;?\[2026-01-27 14:34:23 UTC]  70KiB STANDARD CMMD_clinicaldata_revision.xlsx
[2026-01-27 14:34:23 UTC] 668KiB STANDARD metadata.csv
[2026-01-29 12:26:09 UTC]     0B CMMD/


In [4]:
!mc ls --recursive "s3/dimitri/stat_app/CMMD2022" | grep -E "metadata\.csv|\.xlsx$"


[2026-01-27 14:34:23 UTC]  70KiB STANDARD CMMD_clinicaldata_revision.xlsx
[2026-01-27 14:34:23 UTC] 668KiB STANDARD metadata.csv


In [5]:
!mc mirror --overwrite --debug "s3/dimitri/stat_app/CMMD2022" "./CMMD2022" 2>&1 | grep -E "metadata\.csv|\.xlsx|ERROR|AccessDenied|denied|failed"



`s3/dimitri/stat_app/CMMD2022/CMMD_clinicaldata_revision.xlsx` -> `CMMD2022/CMMD_clinicaldata_revision.xlsx`
`s3/dimitri/stat_app/CMMD2022/metadata.csv` -> `CMMD2022/metadata.csv`
mc: <DEBUG> GET /dimitri/stat_app/CMMD2022/CMMD_clinicaldata_revision.xlsx HTTP/1.1
mc: <DEBUG> GET /dimitri/stat_app/CMMD2022/metadata.csv HTTP/1.1


In [8]:
pip install pydicom openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
import glob
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

import pydicom
try:
    from pydicom.pixel_data_handlers.util import apply_voi_lut
except Exception:
    apply_voi_lut = None


# -----------------------
# CONFIG
# -----------------------
BASE_DIR = "CMMD2022"
DICOM_ROOT = os.path.join(BASE_DIR, "CMMD")
CLINICAL_XLSX = os.path.join(BASE_DIR, "CMMD_clinicaldata_revision.xlsx")

CKPT_PATH = "best_resnet18_miniddsm2_two_classes.pth"
ARCH = "resnet18"        # <-- adapte (resnet50 etc.)
NUM_CLASSES = 2

BATCH_SIZE = 32
NUM_WORKERS = 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 224

# IMPORTANT: mets la même normalisation que pendant ton train.
# Si tu as fine-tuné un ResNet "ImageNet-style", ceci est standard.
TEST_TFMS = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

LABEL_MAP = {"benign": 0, "malignant": 1}


# -----------------------
# MODEL
# -----------------------
def build_model(arch: str, num_classes: int) -> nn.Module:
    if arch == "resnet18":
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if arch == "resnet34":
        m = models.resnet34(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    raise ValueError(f"Unknown arch: {arch}")

def load_checkpoint_into_model(model: nn.Module, ckpt_path: str) -> None:
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt

    # retire "module." si entraîné en DataParallel/DDP
    state = {k.replace("module.", ""): v for k, v in state.items()}

    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print("Missing keys:", missing)
    if unexpected:
        print("Unexpected keys:", unexpected)


# -----------------------
# LABELS: from xlsx
# -----------------------
def build_exam_to_label(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    # normalise textes
    df["ID1"] = df["ID1"].astype(str).str.strip()
    df["classification"] = df["classification"].astype(str).str.strip().str.lower()

    # filtre aux labels attendus
    df = df[df["classification"].isin(LABEL_MAP.keys())].copy()
    df["label"] = df["classification"].map(LABEL_MAP).astype(int)

    # supprime doublons éventuels
    df = df.drop_duplicates(subset=["ID1"], keep="first")

    exam_to_label = dict(zip(df["ID1"], df["label"]))

    print("Répartition labels:", df["label"].value_counts().to_dict())
    print("Nb examens labellisés:", len(exam_to_label))
    return exam_to_label


# -----------------------
# DICOM -> RGB uint8
# -----------------------
def dicom_to_uint8_rgb(ds: pydicom.dataset.FileDataset) -> np.ndarray:
    img = ds.pixel_array

    # VOI LUT utile sur certains DICOM
    if apply_voi_lut is not None:
        try:
            img = apply_voi_lut(img, ds)
        except Exception:
            pass

    img = img.astype(np.float32)

    # rescale slope/intercept si présents
    slope = float(getattr(ds, "RescaleSlope", 1.0))
    intercept = float(getattr(ds, "RescaleIntercept", 0.0))
    img = img * slope + intercept

    # min/max -> [0,1]
    mn, mx = float(img.min()), float(img.max())
    if mx - mn < 1e-6:
        img = np.zeros_like(img)
    else:
        img = (img - mn) / (mx - mn)

    img = (img * 255.0).round().clip(0, 255).astype(np.uint8)
    rgb = np.stack([img, img, img], axis=-1)  # grayscale -> RGB
    return rgb


# -----------------------
# DATASET: 1 exam = 2 dcm
# -----------------------
class CMMDExamDataset(Dataset):
    def __init__(self, dicom_root: str, exam_to_label: dict, tfms):
        self.dicom_root = dicom_root
        self.exam_to_label = exam_to_label
        self.tfms = tfms

        # dossiers D1-* présents localement
        all_exam_dirs = sorted([
            os.path.join(dicom_root, d) for d in os.listdir(dicom_root)
            if os.path.isdir(os.path.join(dicom_root, d)) and d.startswith("D1-")
        ])

        # on garde seulement ceux qui ont un label dans l'xlsx
        self.exam_dirs = [p for p in all_exam_dirs if os.path.basename(p) in exam_to_label]

        if not self.exam_dirs:
            raise RuntimeError("Aucun dossier D1-* avec label trouvé. Vérifie DICOM_ROOT et l'xlsx.")

    def __len__(self):
        return len(self.exam_dirs)

    def __getitem__(self, idx: int):
        exam_dir = self.exam_dirs[idx]
        exam_id = os.path.basename(exam_dir)
        label = self.exam_to_label[exam_id]

        dcm_paths = sorted(glob.glob(os.path.join(exam_dir, "**", "*.dcm"), recursive=True))
        if len(dcm_paths) != 2:
            raise RuntimeError(f"{exam_id}: attendu 2 .dcm, trouvé {len(dcm_paths)}")

        imgs = []
        for p in dcm_paths:
            ds = pydicom.dcmread(p)
            rgb = dicom_to_uint8_rgb(ds)
            t = self.tfms(rgb)
            imgs.append(t)

        return imgs, label, exam_id


def collate_fn(batch):
    x1 = torch.stack([b[0][0] for b in batch], dim=0)
    x2 = torch.stack([b[0][1] for b in batch], dim=0)
    y = torch.tensor([b[1] for b in batch], dtype=torch.long)
    exam_ids = [b[2] for b in batch]
    return x1, x2, y, exam_ids


# -----------------------
# EVAL
# -----------------------
def main():
    exam_to_label = build_exam_to_label(CLINICAL_XLSX)

    ds = CMMDExamDataset(DICOM_ROOT, exam_to_label, TEST_TFMS)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

    model = build_model(ARCH, NUM_CLASSES).to(DEVICE)
    load_checkpoint_into_model(model, CKPT_PATH)
    model.eval()

    correct, total = 0, 0

    with torch.no_grad():
        for x1, x2, y, _ in dl:
            x1 = x1.to(DEVICE, non_blocking=True)
            x2 = x2.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            # 2 DICOM / exam: moyenne des logits
            logits = (model(x1) + model(x2)) / 2.0
            pred = logits.argmax(dim=1)

            correct += (pred == y).sum().item()
            total += y.numel()

    acc = correct / total if total else 0.0
    print(f"Accuracy: {acc:.4f} ({correct}/{total})")


if __name__ == "__main__":
    main()


Répartition labels: {1: 1296, 0: 479}
Nb examens labellisés: 1775


FileNotFoundError: [Errno 2] No such file or directory: 'best_resnet18_miniddsm2_two_classes.pth'